# PCA 

In [ ]:
import pandas as pd
import numpy as np 
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [ ]:
data = pd.read_feather("../data/final_data_ny.feather")

In [ ]:
data 

In [ ]:
y = data[("y_ny", "")]
X = data.drop(columns=["y_ny", "ret_ny"], level=0).select_dtypes("number")
X.columns = ["_".join(filter(None, map(str, c))) for c in X.columns]  # flatten MultiIndex
X = X.replace([np.inf, -np.inf], np.nan).dropna(axis=1, how="all").ffill().dropna()
y = y.loc[X.index]


In [ ]:
# same chronological split as Data_Splitting.ipynb: last 20% is test
# fit scaler + PCA on train only, no look-ahead
TEST_START = X.index[-int(len(data) * 0.2)]
X_train, X_test = X.loc[X.index < TEST_START], X.loc[TEST_START:]
y_train, y_test = y.loc[X_train.index], y.loc[X_test.index]

In [ ]:
scaler = StandardScaler().fit(X_train)
pca = PCA(n_components=0.95).fit(scaler.transform(X_train))  # keep 95% of variance

cols = [f"PC{i+1}" for i in range(pca.n_components_)]
X_train_pca = pd.DataFrame(pca.transform(scaler.transform(X_train)), index=X_train.index, columns=cols)
X_test_pca = pd.DataFrame(pca.transform(scaler.transform(X_test)), index=X_test.index, columns=cols)

print(f"{X.shape[1]} features -> {pca.n_components_} components (95% variance)")
print("train:", X_train_pca.shape, "test:", X_test_pca.shape)
X_train_pca.head()

In [ ]:
pca

In [ ]:
plt.plot(np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.axhline(0.95, ls="--", c="grey")
plt.xlabel("components"); plt.ylabel("cumulative explained variance")
plt.show()

# top features driving each of the first 3 components
loadings = pd.DataFrame(pca.components_.T, index=X.columns, columns=cols)
for pc in cols[:30]:
    print(pc, loadings[pc].abs().nlargest(5).index.tolist())


In [ ]:
# PCA already sorts components by explained variance, so the top 20 are PC1..PC20
top20 = cols[:20]
print(f"top ... explain {pca.explained_variance_ratio_[:55].sum():.1%} of variance")


In [ ]:
loadings["PC1"].sort_values(key=abs,ascending=False)

In [ ]:
top_w = loadings[cols[:20]].abs()
useless = top_w.index[(top_w < 0.05).all(axis=1)]   # weak in every kept PC
print(len(useless), useless.tolist())